# Universidad Sergio Arboleda
## Maestría en Inteligencia Artificial - Módulo 1
### Taller 4: Comparación Experimental de Metaheurísticas y Algoritmos Bioinspirados sobre el Problema del Viajante de Comercio (TSP)

**Autores:** Bryan Orozco Romero y Equipo  
**Algoritmos Evaluados:**
1. Hill Climbing (HC) con reinicios aleatorios
2. Simulated Annealing (SA) con enfriamiento geométrico y regla de Metropolis
3. Genetic Algorithm (GA) con Order Crossover (OX) y selección por torneo
4. Ant Colony Optimization (ACO / Ant System) con matriz de feromonas y visibilidad
5. Particle Swarm Optimization (PSO) con Smallest Position Value (SPV / Random Keys)

**Objetivo:** Evaluar el desempeño computacional y la calidad de solución bajo un protocolo estricto de **Evaluaciones de la Función Objetivo (FEs)**, controlando variables y midiendo tiempo y memoria RAM.

---
## 1. Configuración de Entorno e Importación de Módulos

In [ ]:
import sys
import os
import time
import tracemalloc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Asegurar importación de la carpeta src
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from src.tsp import generar_instancia_tsp, calcular_costo_ruta, es_ruta_valida
from src.base import EvaluadorTSP, generar_vecino_2opt
from src import hill_climbing
from src import simulated_annealing
from src import genetic_algorithm
from src import aco
from src import pso

print("[OK] Modulos y librerias cargados exitosamente.")

---
## 2. Definición del Problema: Instancia TSP y Matriz Euclidiana
El Problema del Viajante de Comercio (TSP) busca encontrar la permutación cíclica $\pi = (\pi_0, \pi_1, \dots, \pi_{n-1})$ de $n$ ciudades que minimice el costo del recorrido cerrado:
$$f(\pi) = \sum_{i=0}^{n-2} d(\pi_i, \pi_{i+1}) + d(\pi_{n-1}, \pi_0)$$

In [ ]:
# Generamos una instancia de prueba de 20 ciudades con semilla controlada
n_ciudades = 20
semilla_instancia = 42

coords, distancias = generar_instancia_tsp(n_ciudades, semilla=semilla_instancia)

print(f"Dimensiones de coordenadas: {coords.shape}")
print(f"Matriz de distancias: {distancias.shape}")
print(f"Distancia entre ciudad 0 y 1: {distancias[0, 1]:.2f}")

---
## 3. Validación Formal del Punto 1 (Interfaz Común)
Cada algoritmo implementa estrictamente la firma común:
`optimizar(distancias, presupuesto, semilla, parametros=None)`
retornando:
- `mejor_ruta`: Permutación válida de ciudades
- `mejor_costo`: Valor recalculado y consistente
- `evaluaciones`: FEs utilizadas <= presupuesto
- `tiempo_s`: Tiempo de ejecución en segundos
- `memoria_mb`: Memoria pico consumida en MB
- `historial`: Hitos de convergencia al 1%, 5%, 10%, 20%, 40%, 60%, 80%, 100%

In [ ]:
presupuesto = 1000
semilla = 12345

algoritmos = [
    ("Hill Climbing", hill_climbing),
    ("Simulated Annealing", simulated_annealing),
    ("Genetic Algorithm", genetic_algorithm),
    ("Ant Colony Optimization", aco),
    ("Particle Swarm Optimization", pso)
]

resultados_validacion = []

for nombre, mod in algoritmos:
    res = mod.optimizar(distancias, presupuesto, semilla)
    
    # Aserciones formales
    assert es_ruta_valida(res["mejor_ruta"], n_ciudades), f"Ruta invalida en {nombre}"
    recalc = calcular_costo_ruta(res["mejor_ruta"], distancias)
    assert np.isclose(res["mejor_costo"], recalc, atol=1e-5), f"Inconsistencia de costo en {nombre}"
    assert res["evaluaciones"] <= presupuesto, f"Presupuesto excedido en {nombre}"
    
    resultados_validacion.append({
        "Algoritmo": nombre,
        "Mejor Costo": round(res["mejor_costo"], 2),
        "FEs": res["evaluaciones"],
        "Tiempo (s)": round(res["tiempo_s"], 4),
        "Memoria (MB)": res["memoria_mb"],
        "Hitos Historial": len(res["historial"])
    })

df_val = pd.DataFrame(resultados_validacion)
display(df_val)

---
## 4. Visualización de Rutas 2D
Comparación visual de las rutas encontradas por los 5 algoritmos sobre el plano bidimensional $[0, 100] \times [0, 100]$.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle(f"Rutas Finales TSP (n={n_ciudades} Ciudades) - Comparativa de Metaheuristicas", fontsize=16, fontweight="bold", y=1.02)

colores = ["#e74c3c", "#e67e22", "#2ecc71", "#1abc9c", "#3498db"]

for idx, (nombre, mod) in enumerate(algoritmos):
    ax = axes[idx]
    res = mod.optimizar(distancias, presupuesto=2000, semilla=12345)
    ruta = res["mejor_ruta"]
    costo = res["mejor_costo"]
    
    # Coordenadas ordenadas segun la ruta circular
    orden = ruta + [ruta[0]]
    coords_orden = coords[orden]
    
    ax.plot(coords_orden[:, 0], coords_orden[:, 1], color=colores[idx], lw=1.8, alpha=0.85, zorder=2)
    ax.scatter(coords[:, 0], coords[:, 1], color="#2c3e50", s=45, zorder=3)
    ax.scatter(coords[ruta[0], 0], coords[ruta[0], 1], color="#f1c40f", s=110, edgecolor="black", zorder=4, label="Inicio")
    
    ax.set_title(f"{nombre}\nCosto: {costo:.2f}", fontsize=11, fontweight="bold")
    ax.set_xlim(-5, 105)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

---
## 5. Curvas de Convergencia (FEs vs. Mejor Costo)
Se analiza la evolución de la mejor solución encontrada a lo largo del presupuesto de evaluaciones de función.

In [ ]:
plt.figure(figsize=(10, 6))

for idx, (nombre, mod) in enumerate(algoritmos):
    res = mod.optimizar(distancias, presupuesto=3000, semilla=12345)
    historial = res["historial"]
    fes = [h[0] for h in historial]
    costos = [h[1] for h in historial]
    plt.plot(fes, costos, marker="o", label=nombre, color=colores[idx], lw=2)

plt.xlabel("Evaluaciones de la Función Objetivo (FEs)", fontsize=12, fontweight="bold")
plt.ylabel("Mejor Costo Encontrado (Longitud del Tour)", fontsize=12, fontweight="bold")
plt.title(f"Curvas de Convergencia de las 5 Metaheuristicas (TSP n={n_ciudades})", fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 6. Resultados Experimentales Consolidados
Carga y visualización de la tabla estadística generada por el protocolo experimental oficial.

In [ ]:
ruta_csv = os.path.join("..", "resultados", "resumen_metricas.csv")
if not os.path.exists(ruta_csv):
    ruta_csv = os.path.join("resultados", "resumen_metricas.csv")

if os.path.exists(ruta_csv):
    df_resumen = pd.read_csv(ruta_csv)
    print("Tabla Oficial de Metricas Experimentales:")
    display(df_resumen)
else:
    print("Ejecuta 'python ejecutar_experimentos.py --config configuracion.json' para generar la tabla consolidada.")

---
## 7. Conclusiones y Discusión Crítica

1. **Superioridad de Ant Colony Optimization (ACO) en Grafos:**  
   ACO domina claramente en instancias más grandes ($n=50$) alcanzando una tasa de éxito del 100% y un error relativo medio de solo 0.15%. Esto se debe a que la matriz de feromonas $\tau_{ij}$ mapea de forma directa y natural la topología de aristas del grafo.

2. **Efectividad Local de Hill Climbing (HC) con 2-opt:**  
   Gracias al operador 2-opt, HC desenreda eficientemente los cruces de aristas en el plano euclidiano en tiempos ultrarrápidos (< 0.5s), superando a métodos más complejos en instancias pequeñas.

3. **Dificultad de PSO en Problemas Combinatorios:**  
   PSO presenta mayor dispersión y costo promedio debido al mapeo continuo $\to$ discreto mediante Random Keys / SPV (`np.argsort`), el cual introduce distorsiones en la preservación de orden topológico en espacios de permutaciones.

4. **Equidad Experimental por FEs:**  
   El uso estricto de la clase `EvaluadorTSP` garantiza que ningún algoritmo reciba más llamadas de evaluación que otro, aislando la calidad del operador de búsqueda de la capacidad computacional.